In [141]:
# ===============================================================
# 0. IMPORTS
# ===============================================================
import numpy as np
import pandas as pd
import itertools
import torch

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping as PlEarlyStopping
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import RMSE
from pytorch_forecasting.data import GroupNormalizer

import mlflow

np.random.seed(42)
torch.manual_seed(42)

# ===============================================================
# 1. GLOBAL MLFLOW CONFIG — set ONCE
# ===============================================================
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("AgroFood_CO2_Emissions")

# ===============================================================
# 2. LOAD PRE-SPLIT TRAIN/TEST (SAME AS BASELINE MODELS & LSTM)
# ===============================================================
df_train = pd.read_csv("agro_co2_train.csv")
df_test  = pd.read_csv("agro_co2_test.csv")

TRAIN_END_YEAR = df_train["Year"].max()

# Combine for SAFE feature engineering
df_all = pd.concat([df_train, df_test], ignore_index=True)
df_all = df_all.sort_values(["Area", "Year"]).reset_index(drop=True)

# ===============================================================
# 3. FEATURE BUCKETS (same as baseline & LSTM)
# ===============================================================
land_use_features = [
    "Savanna fires", "Forest fires", "Forestland",
    "Net Forest conversion", "Drained organic soils (CO2)",
    "Fires in organic soils", "Fires in humid tropical forests"
]

ag_features = [
    "Crop Residues", "Rice Cultivation", "Manure applied to Soils",
    "Manure left on Pasture", "Manure Management",
    "Fertilizers Manufacturing", "IPPU"
]

energy_features = [
    "On-farm Electricity Use", "Food Processing", "Food Packaging",
    "Food Transport", "Food Retail", "Food Household Consumption",
    "Agrifood Systems Waste Disposal", "On-farm energy use",
    "Pesticides Manufacturing"
]

population_features = [
    "Rural population", "Urban population",
    "Total Population - Male", "Total Population - Female"
]

temperature_features = ["Average Temperature °C"]
time_features = ["Year"]

numerical_drivers = (
      land_use_features
    + ag_features
    + energy_features
    + population_features
    + temperature_features
    + time_features
)

categorical_features = ["Area"]
target_col = "log_total_emission"


# ===============================================================
# 4. TARGET ENGINEERING (ensure log target exists)
# ===============================================================
if "log_total_emission" not in df_all:
    emission_min = df_all["total_emission"].min()
    shift_amount = 1 - emission_min if emission_min <= 0 else 0
    df_all["total_emission_shifted"] = df_all["total_emission"] + shift_amount
    df_all["log_total_emission"] = np.log(df_all["total_emission_shifted"])


# ===============================================================
# 5. TIME FEATURES: lag1, lag2, roll3, delta
# ===============================================================
def add_time_features(df):
    df = df.sort_values(["Area", "Year"])
    df["lag1"] = df.groupby("Area")[target_col].shift(1)
    df["lag2"] = df.groupby("Area")[target_col].shift(2)

    df["roll3"] = (
        df.groupby("Area")[target_col]
            .rolling(3, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
    )

    df["delta"] = df[target_col] - df["lag1"]
    return df

df_all = add_time_features(df_all)


# ===============================================================
# 6. SAFELY FILL LAGS & ROLLING (CRITICAL FIX)
# ===============================================================
def fill_group_na(df, cols, group="Area"):
    for col in cols:
        df[col] = (
            df.groupby(group)[col]
              .apply(lambda s: s.fillna(method="ffill").fillna(method="bfill"))
              .fillna(0)              # final fallback required by TFT
        )
    return df

df_all = fill_group_na(df_all, ["lag1", "lag2", "roll3", "delta"])

# FINAL NA CLEANUP FOR TFT
df_all = df_all.replace([np.inf, -np.inf], 0)
df_all = df_all.fillna(0)

engineered_features = ["lag1", "lag2", "roll3", "delta"]
tft_numerical = numerical_drivers + engineered_features


# ===============================================================
# 7. RE-SPLIT ENGINEERED DATA
# ===============================================================
train_tft = df_all[df_all["Year"] <= TRAIN_END_YEAR].copy()
test_tft  = df_all[df_all["Year"] > TRAIN_END_YEAR].copy()

# Add TFT-required fields
train_tft["time_idx"] = train_tft.groupby("Area").cumcount()
test_tft["time_idx"]  = test_tft.groupby("Area").cumcount()

# group_id MUST BE STRING — FIX #2
train_tft["group_id"] = train_tft["Area"].astype(str)
test_tft["group_id"]  = test_tft["Area"].astype(str)

# Known / unknown reals
known_real_features = ["Year"]
unknown_real_features = tft_numerical


# ===============================================================
# 8. TFT HYPERPARAMETER SEARCH LOOP
# ===============================================================
hidden_sizes = [16, 32]
hidden_cont_sizes = [8, 16]
dropouts = [0.1, 0.3]
lrs = [0.003, 0.01]
encoder_lengths = [4, 6]

batch_size = 64
max_epochs = 50

results = []

for hs, hc, dr, lr, enc_len in itertools.product(
    hidden_sizes, hidden_cont_sizes, dropouts, lrs, encoder_lengths
):

    print(f"\n=== TFT Trial: hs={hs}, hc={hc}, dr={dr}, lr={lr}, enc={enc_len} ===")

    # ===============================================================
    # 8A. BUILD TFT DATASETS (NO MLflow)
    # ===============================================================
    target_normalizer = GroupNormalizer(groups=["group_id"])

    training_data = TimeSeriesDataSet(
        train_tft,
        time_idx="time_idx",
        target=target_col,
        group_ids=["group_id"],
        max_encoder_length=enc_len,
        min_encoder_length=max(2, enc_len // 2),
        max_prediction_length=3,
        static_categoricals=["group_id"],
        time_varying_known_reals=known_real_features,
        time_varying_unknown_reals=unknown_real_features + [target_col],
        target_normalizer=target_normalizer,
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )

    validation_data = TimeSeriesDataSet.from_dataset(
        training_data,
        test_tft,
        predict=True,
        stop_randomization=True,
    )

    train_dl = training_data.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_dl   = validation_data.to_dataloader(train=False, batch_size=batch_size, num_workers=0)


    # ===============================================================
    # 8B. MLFLOW RUN STARTS HERE
    # ===============================================================
    with mlflow.start_run(run_name="TFT_Tuning") as run:

        # -----------------------------------
        # Build TFT model
        # -----------------------------------
        tft = TemporalFusionTransformer.from_dataset(
            training_data,
            learning_rate=lr,
            hidden_size=hs,
            hidden_continuous_size=hc,
            attention_head_size=1,
            dropout=dr,
            loss=RMSE(),
            reduce_on_plateau_patience=3,
        )

        # -----------------------------------
        # Trainer
        # -----------------------------------
        trainer = pl.Trainer(
            max_epochs=max_epochs,
            accelerator="cpu",
            callbacks=[
                PlEarlyStopping(
                    monitor="val_loss",
                    patience=5,
                    min_delta=1e-4,
                    mode="min"
                )
            ],
            enable_progress_bar=False,
        )

        # Log hyperparams
        mlflow.log_param("hidden_size", hs)
        mlflow.log_param("hidden_cont", hc)
        mlflow.log_param("dropout", dr)
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("encoder_length", enc_len)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("max_epochs", max_epochs)

        # Fit model
        trainer.fit(tft, train_dl, val_dl)

        # Load best model
        best_path = trainer.checkpoint_callback.best_model_path
        best_tft = TemporalFusionTransformer.load_from_checkpoint(best_path)

        # ===========================================================
        # 8C. EVALUATE
        # ===========================================================
        with torch.no_grad():
            preds = best_tft.predict(val_dl).flatten()

        actuals = torch.cat([y[0] for _, y in val_dl]).flatten()

        mse = torch.mean((actuals - preds)**2)
        rmse = torch.sqrt(mse).item()

        ss_res = torch.sum((actuals - preds)**2)
        ss_tot = torch.sum((actuals - actuals.mean())**2)
        r2 = (1 - ss_res / ss_tot).item()

        print(f"→ Test RMSE: {rmse:.4f}, R²: {r2:.4f}")

        mlflow.log_metric("rmse_test", rmse)
        mlflow.log_metric("r2_test", r2)

        # Save results
        results.append({
            "hidden_size": hs,
            "hidden_cont": hc,
            "dropout": dr,
            "learning_rate": lr,
            "encoder_length": enc_len,
            "rmse": rmse,
            "r2": r2,
            "run_id": run.info.run_id
        })


# ===============================================================
# DONE — Show best 10 configs
# ===============================================================
results_df = pd.DataFrame(results)
results_df.sort_values("rmse").head(10)


/var/folders/1y/b256x0zs517c5bwmlhdp0pgc0000gn/T/ipykernel_67901/1227523314.py:119: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda s: s.fillna(method="ffill").fillna(method="bfill"))
/var/folders/1y/b256x0zs517c5bwmlhdp0pgc0000gn/T/ipykernel_67901/1227523314.py:119: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .gr


=== TFT Trial: hs=16, hc=8, dr=0.1, lr=0.003, enc=4 ===


Exception: Run with UUID e28bba30dd034f39877bd44a1559ac15 is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

## WITHOUT TEMPERATURE FEATURE

In [ ]:
# ===============================================================
# 0. IMPORTS
# ===============================================================
import numpy as np
import pandas as pd
import itertools
import torch

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping as PlEarlyStopping
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import RMSE
from pytorch_forecasting.data import GroupNormalizer

import mlflow

np.random.seed(42)
torch.manual_seed(42)

# ===============================================================
# 1. GLOBAL MLFLOW CONFIG — set ONCE
# ===============================================================
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("AgroFood_CO2_Emissions")

# ===============================================================
# 2. LOAD PRE-SPLIT TRAIN/TEST (SAME AS BASELINE MODELS & LSTM)
# ===============================================================
df_train = pd.read_csv("agro_co2_train.csv")
df_test  = pd.read_csv("agro_co2_test.csv")

TRAIN_END_YEAR = df_train["Year"].max()

# Combine for SAFE feature engineering
df_all = pd.concat([df_train, df_test], ignore_index=True)
df_all = df_all.sort_values(["Area", "Year"]).reset_index(drop=True)

# ===============================================================
# 3. FEATURE BUCKETS (same as baseline & LSTM)
# ===============================================================
land_use_features = [
    "Savanna fires", "Forest fires", "Forestland",
    "Net Forest conversion", "Drained organic soils (CO2)",
    "Fires in organic soils", "Fires in humid tropical forests"
]

ag_features = [
    "Crop Residues", "Rice Cultivation", "Manure applied to Soils",
    "Manure left on Pasture", "Manure Management",
    "Fertilizers Manufacturing", "IPPU"
]

energy_features = [
    "On-farm Electricity Use", "Food Processing", "Food Packaging",
    "Food Transport", "Food Retail", "Food Household Consumption",
    "Agrifood Systems Waste Disposal", "On-farm energy use",
    "Pesticides Manufacturing"
]

population_features = [
    "Rural population", "Urban population",
    "Total Population - Male", "Total Population - Female"
]

# temperature_features = ["Average Temperature °C"]
time_features = ["Year"]

numerical_drivers = (
      land_use_features
    + ag_features
    + energy_features
    + population_features
    # + temperature_features
    + time_features
)

categorical_features = ["Area"]
target_col = "log_total_emission"


# ===============================================================
# 4. TARGET ENGINEERING (ensure log target exists)
# ===============================================================
if "log_total_emission" not in df_all:
    emission_min = df_all["total_emission"].min()
    shift_amount = 1 - emission_min if emission_min <= 0 else 0
    df_all["total_emission_shifted"] = df_all["total_emission"] + shift_amount
    df_all["log_total_emission"] = np.log(df_all["total_emission_shifted"])


# ===============================================================
# 5. TIME FEATURES: lag1, lag2, roll3, delta
# ===============================================================
# Helps in understandong the past few years data (emission from last year, emission from two years ago, average emission over last 3 years, change from last year)
# delta gives yoy change
def add_time_features(df):
    df = df.sort_values(["Area", "Year"])
    df["lag1"] = df.groupby("Area")[target_col].shift(1)
    df["lag2"] = df.groupby("Area")[target_col].shift(2)

    df["roll3"] = (
        df.groupby("Area")[target_col]
            .rolling(3, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
    )

    df["delta"] = df[target_col] - df["lag1"]
    return df

df_all = add_time_features(df_all)


# ===============================================================
# 6. SAFELY FILL LAGS & ROLLING (CRITICAL FIX)
# ===============================================================
def fill_group_na(df, cols, group="Area"):
    for col in cols:
        df[col] = (
            df.groupby(group)[col]
              .apply(lambda s: s.fillna(method="ffill").fillna(method="bfill"))
              .fillna(0)              # final fallback required by TFT
        )
    return df

df_all = fill_group_na(df_all, ["lag1", "lag2", "roll3", "delta"])

# FINAL NA CLEANUP FOR TFT
df_all = df_all.replace([np.inf, -np.inf], 0)
df_all = df_all.fillna(0)

engineered_features = ["lag1", "lag2", "roll3", "delta"]
tft_numerical = numerical_drivers + engineered_features


# ===============================================================
# 7. RE-SPLIT ENGINEERED DATA
# ===============================================================
# Each country is a separate time series 
#time_idx is a monotonically increasing index per country
train_tft = df_all[df_all["Year"] <= TRAIN_END_YEAR].copy()
test_tft  = df_all[df_all["Year"] > TRAIN_END_YEAR].copy()

# Add TFT-required fields
train_tft["time_idx"] = train_tft.groupby("Area").cumcount()
test_tft["time_idx"]  = test_tft.groupby("Area").cumcount()

# group_id MUST BE STRING — FIX #2
train_tft["group_id"] = train_tft["Area"].astype(str)
test_tft["group_id"]  = test_tft["Area"].astype(str)

# Known / unknown reals
known_real_features = ["Year"]
unknown_real_features = tft_numerical


# ===============================================================
# 8. TFT HYPERPARAMETER SEARCH LOOP
# ===============================================================
hidden_sizes = [16, 32]
hidden_cont_sizes = [8, 16]
dropouts = [0.1, 0.3]
lrs = [0.003, 0.01]
encoder_lengths = [4, 6]

batch_size = 64
max_epochs = 50

results = []

for hs, hc, dr, lr, enc_len in itertools.product(
    hidden_sizes, hidden_cont_sizes, dropouts, lrs, encoder_lengths
):

    print(f"\n=== TFT Trial: hs={hs}, hc={hc}, dr={dr}, lr={lr}, enc={enc_len} ===")

    # ===============================================================
    # 8A. BUILD TFT DATASETS (NO MLflow)
    # ===============================================================
    # encoder window - history, decoder window - forecast
    target_normalizer = GroupNormalizer(groups=["group_id"])

    training_data = TimeSeriesDataSet(
        train_tft,
        time_idx="time_idx",
        target=target_col,
        group_ids=["group_id"],
        max_encoder_length=enc_len,
        min_encoder_length=max(2, enc_len // 2),
        max_prediction_length=3,
        static_categoricals=["group_id"],
        time_varying_known_reals=known_real_features,
        time_varying_unknown_reals=unknown_real_features + [target_col],
        target_normalizer=target_normalizer,
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )

    validation_data = TimeSeriesDataSet.from_dataset(
        training_data,
        test_tft,
        predict=True,
        stop_randomization=True,
    )

    train_dl = training_data.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_dl   = validation_data.to_dataloader(train=False, batch_size=batch_size, num_workers=0)


    # ===============================================================
    # 8B. MLFLOW RUN STARTS HERE
    # ===============================================================
    with mlflow.start_run(run_name="TFT_Tuning") as run:

        # -----------------------------------
        # Build TFT model
        # -----------------------------------
        tft = TemporalFusionTransformer.from_dataset(
            training_data,
            learning_rate=lr,
            hidden_size=hs,
            hidden_continuous_size=hc,
            attention_head_size=1,
            dropout=dr,
            loss=RMSE(),
            reduce_on_plateau_patience=3,
        )

        # -----------------------------------
        # Trainer
        # -----------------------------------
        trainer = pl.Trainer(
            max_epochs=max_epochs,
            accelerator="cpu",
            callbacks=[
                PlEarlyStopping(
                    monitor="val_loss",
                    patience=5,
                    min_delta=1e-4,
                    mode="min"
                )
            ],
            enable_progress_bar=False,
        )

        # Log hyperparams
        mlflow.log_param("hidden_size", hs)
        mlflow.log_param("hidden_cont", hc)
        mlflow.log_param("dropout", dr)
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("encoder_length", enc_len)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("max_epochs", max_epochs)

        # Fit model
        trainer.fit(tft, train_dl, val_dl)

        # Load best model
        best_path = trainer.checkpoint_callback.best_model_path
        best_tft = TemporalFusionTransformer.load_from_checkpoint(best_path)

        # ===========================================================
        # 8C. EVALUATE
        # ===========================================================
        with torch.no_grad():
            preds = best_tft.predict(val_dl).flatten()

        actuals = torch.cat([y[0] for _, y in val_dl]).flatten()

        mse = torch.mean((actuals - preds)**2)
        rmse = torch.sqrt(mse).item()

        ss_res = torch.sum((actuals - preds)**2)
        ss_tot = torch.sum((actuals - actuals.mean())**2)
        r2 = (1 - ss_res / ss_tot).item()

        print(f"→ Test RMSE: {rmse:.4f}, R²: {r2:.4f}")

        mlflow.log_metric("rmse_test", rmse)
        mlflow.log_metric("r2_test", r2)

        # Save results
        results.append({
            "hidden_size": hs,
            "hidden_cont": hc,
            "dropout": dr,
            "learning_rate": lr,
            "encoder_length": enc_len,
            "rmse": rmse,
            "r2": r2,
            "run_id": run.info.run_id
        })


# ===============================================================
# DONE — Show best 10 configs
# ===============================================================
results_df = pd.DataFrame(results)
results_df.sort_values("rmse").head(10)


/var/folders/1y/b256x0zs517c5bwmlhdp0pgc0000gn/T/ipykernel_67901/2356090828.py:119: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda s: s.fillna(method="ffill").fillna(method="bfill"))
/var/folders/1y/b256x0zs517c5bwmlhdp0pgc0000gn/T/ipykernel_67901/2356090828.py:119: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .gr


=== TFT Trial: hs=16, hc=8, dr=0.1, lr=0.003, enc=4 ===


/opt/anaconda3/envs/acad/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:269: UserWarning: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
  rank_zero_warn(
/opt/anaconda3/envs/acad/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:269: UserWarning: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
  rank_zero_warn(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/opt/anaconda3/envs/acad/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  ra

→ Test RMSE: 0.1148, R²: 0.8241
🏃 View run TFT_Tuning at: http://127.0.0.1:5000/#/experiments/965617254655966202/runs/d217afec8122495088d7fd56e07f8e5a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/965617254655966202

=== TFT Trial: hs=16, hc=8, dr=0.1, lr=0.003, enc=6 ===


/opt/anaconda3/envs/acad/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | RMSE                            | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 3.8 K 
3  | prescalers                         | ModuleDict                      | 592   
4  | static_variable_selection          | VariableSelectionNetwork        | 1.9 K 
5  | encoder_variable_selection         | VariableSelectionNetwork        | 24.2 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 1.2 K 
7  | static_co

...

/opt/anaconda3/envs/acad/lib/python3.10/site-packages/pytorch_forecasting/data/timeseries.py:1282: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 7 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__group_id': 'Czechoslovakia'}, {'__group_id__group_id': 'Ethiopia PDR'}, {'__group_id__group_id': 'Pacific Islands Trust Territory'}, {'__group_id__group_id': 'South Sudan'}, {'__group_id__group_id': 'Sudan'}, {'__group_id__group_id': 'USSR'}, {'__group_id__group_id': 'Yugoslav SFR'}]
  warnings.warn(
/opt/anaconda3/envs/acad/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:269: UserWarning: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
  rank_zero_warn(
/opt/anacond

→ Test RMSE: 0.0784, R²: 0.9179
🏃 View run TFT_Tuning at: http://127.0.0.1:5000/#/experiments/965617254655966202/runs/f11c93d6412a440abdca1cab34522876
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/965617254655966202

=== TFT Trial: hs=32, hc=16, dr=0.3, lr=0.01, enc=6 ===


/opt/anaconda3/envs/acad/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:224: PossibleUserWarning: The dataloader, val_dataloader 0, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 8 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(
/opt/anaconda3/envs/acad/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:224: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 8 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(
/opt/anaconda3/envs/acad/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:269: UserWarning: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpoi

→ Test RMSE: 0.0643, R²: 0.9449
🏃 View run TFT_Tuning at: http://127.0.0.1:5000/#/experiments/965617254655966202/runs/0555979b0d28423e879a4f053e507421
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/965617254655966202


,hidden_size,hidden_cont,dropout,learning_rate,encoder_length,rmse,r2,run_id
18,32,8,0.1,0.010,4,0.039808,0.978848,346f7c93fe67423898b034953c6af256
7,16,8,0.3,0.010,6,0.054090,0.960947,35926eaa751944bc96897a67a00a139f
24,32,16,0.1,0.003,4,0.054649,0.960136,4f261896990d48ebaadfe93094aab7f7
5,16,8,0.3,0.003,6,0.055478,0.958917,05194a725c104b2a8a24fd932cbea405
28,32,16,0.3,0.003,4,0.056393,0.957551,d5e4d213799c406a9f2f6c5433894994
22,32,8,0.3,0.010,4,0.058809,0.953837,fa3377dd302e41da9eac1104b5ed2a8c
2,16,8,0.1,0.010,4,0.059632,0.952535,1723c0bc9bbf4464b9b7e67aa722c620
20,32,8,0.3,0.003,4,0.059939,0.952045,3def21cc2a78466d9028b6ce970cab7d
9,16,16,0.1,0.003,6,0.060388,0.951323,ceab5cc2f03147e49d7cbfd449b8f88a
14,16,16,0.3,0.010,4,0.063559,0.946077,2dcd2a9801cb45979fb0082b7b2a616a
